<a href="https://colab.research.google.com/github/DQuanBui/Workforce_Performance_Analysis/blob/main/workforce_performance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workforce Performance Analysis

## Setup the libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

In [3]:
FILE = "employee_performance.xlsx"
sheets = pd.read_excel(FILE, sheet_name=None)   # dict of {sheet_name: DataFrame}
print("Sheets:", list(sheets.keys()))

employees = sheets["employees"]
monthly   = sheets["monthly_performance"]
kpis      = sheets["role_kpis"]

employees.head()

Sheets: ['employees', 'stores', 'monthly_performance', 'role_kpis', 'business_outcomes', 'Data_Dictionary']


,Employee_Id,Full_Name,Age,Education_Level,Hire_Date,Exit_Date,Department,Job_Role,Job_Level,Employment_Type,Base_Salary_Annual,Store_Location,Store_Location_Latitude,Store_Location_Longitude,Store_Id,Manager_Id,Manager_Name,Manager_Status
0,EMP000001,Marie Kim DDS,23,Master's,26/04/2021,13/04/2022,Store Operations,Cashier,Entry,Full-time,25745.40,Charlotte,35.2271,-80.8431,STR058,EMP001131,Michelle Walls,Executive
1,EMP000002,Danny Morgan,26,Master's,03/09/2021,NaN,Store Operations,Sales Associate,Associate,Full-time,39408.57,San Jose,37.3382,-121.8863,STR137,EMP002456,Jennifer Moore,Executive
2,EMP000003,Crystal Robinson,38,Bachelor's,25/08/2016,NaN,Store Operations,Cashier,Entry,Full-time,29319.94,Jacksonville,30.3322,-81.6557,STR008,EMP000529,Melissa Bishop,Senior Manager
3,EMP000004,Mark Perez,33,Bachelor's,10/12/2021,NaN,Store Operations,Senior Sales Associate,Senior_Associate,Full-time,43979.88,Las Vegas,36.1699,-115.1398,STR063,EMP001305,Tina Walter,Senior Manager
4,EMP000005,Shannon Jones,24,High School,24/02/2018,NaN,Store Operations,Cashier,Entry,Full-time,23560.19,Chicago,41.8781,-87.6298,STR109,EMP001823,Nicholas Fitzgerald,Senior Manager


In [4]:
for name, frame in sheets.items():
    print(f"{name:22s} {frame.shape}")

employees              (7500, 18)
stores                 (150, 7)
monthly_performance    (236591, 13)
role_kpis              (236591, 9)
business_outcomes      (16200, 9)
Data_Dictionary        (69, 5)


In [6]:
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Employee_Id               7500 non-null   object 
 1   Full_Name                 7500 non-null   object 
 2   Age                       7500 non-null   int64  
 3   Education_Level           7500 non-null   object 
 4   Hire_Date                 7500 non-null   object 
 5   Exit_Date                 1491 non-null   object 
 6   Department                7500 non-null   object 
 7   Job_Role                  7500 non-null   object 
 8   Job_Level                 7500 non-null   object 
 9   Employment_Type           7500 non-null   object 
 10  Base_Salary_Annual        7500 non-null   float64
 11  Store_Location            7500 non-null   object 
 12  Store_Location_Latitude   7500 non-null   float64
 13  Store_Location_Longitude  7500 non-null   float64
 14  Store_Id

In [7]:
employees.describe()

,Age,Base_Salary_Annual,Store_Location_Latitude,Store_Location_Longitude
count,7500.000000,7500.000000,7500.000000,7500.000000
mean,32.450000,27253.855732,36.954572,-96.160918
std,7.482319,10217.242704,4.878566,16.457037
min,21.000000,8823.460000,29.424100,-122.678400
25%,27.000000,19899.520000,32.776700,-112.074000
50%,32.000000,27394.165000,36.169900,-95.369800
75%,37.000000,31945.697500,39.961200,-82.998800
max,64.000000,177873.610000,47.606200,-71.058900


In [8]:
employees.isna().sum()

,0
Employee_Id,0
Full_Name,0
Age,0
Education_Level,0
Hire_Date,0
Exit_Date,6009
Department,0
Job_Role,0
Job_Level,0
Employment_Type,0


In [9]:
employees["Hire_Date"] = pd.to_datetime(employees["Hire_Date"], format="%d/%m/%Y")
employees["Exit_Date"] = pd.to_datetime(employees["Exit_Date"], format="%d/%m/%Y")

# Target
employees["Attrition"] = employees["Exit_Date"].notna().astype(int)

# Tenure: leavers -> hire..exit, active -> hire..snapshot
snapshot = pd.Timestamp("2024-12-31")
end_date = employees["Exit_Date"].fillna(snapshot)
employees["Tenure_Years"] = ((end_date - employees["Hire_Date"]).dt.days / 365.25).round(2)

print("Overall attrition rate: {:.1f}%".format(employees["Attrition"].mean() * 100))
employees[["Employee_Id","Hire_Date","Exit_Date","Attrition","Tenure_Years"]].head()

Overall attrition rate: 19.9%


,Employee_Id,Hire_Date,Exit_Date,Attrition,Tenure_Years
0,EMP000001,2021-04-26,2022-04-13,1,0.96
1,EMP000002,2021-09-03,NaT,0,3.33
2,EMP000003,2016-08-25,NaT,0,8.35
3,EMP000004,2021-12-10,NaT,0,3.06
4,EMP000005,2018-02-24,NaT,0,6.85
